In [95]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arsalan9702/tickharm-pre-processed-audio")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/arsalan9702/tickharm-pre-processed-audio


In [96]:
path = kagglehub.dataset_download("aryansraut/preprocessed-ucf-crime-dataset-visual")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/aryansraut/preprocessed-ucf-crime-dataset-visual


In [97]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [98]:
!pip install torchlibrosa

In [99]:
from torchlibrosa.stft import Spectrogram, LogmelFilterBank

class CNN14(nn.Module):
    def __init__(self, classes_num=4):
        super().__init__()

        self.spectrogram_extractor = Spectrogram(
            n_fft=1024, hop_length=320, win_length=1024,
            window='hann', center=True, pad_mode='reflect'
        )

        self.logmel_extractor = LogmelFilterBank(
            sr=16000, n_fft=1024, n_mels=64,
            fmin=50, fmax=8000
        )

        self.bn0 = nn.BatchNorm2d(64)

        self.conv_block1 = self._conv_block(1, 64)
        self.conv_block2 = self._conv_block(64, 128)
        self.conv_block3 = self._conv_block(128, 256)
        self.conv_block4 = self._conv_block(256, 512)

        self.fc1 = nn.Linear(512, 512)
        self.fc_out = nn.Linear(512, classes_num)

    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)
        )

    def forward(self, x):
        x = self.spectrogram_extractor(x)
        x = self.logmel_extractor(x)
        x = x.transpose(1, 3)
        x = self.bn0(x)
        x = x.transpose(1, 3)

        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)

        x = torch.mean(x, dim=3)
        x = torch.mean(x, dim=2)

        x = F.relu(self.fc1(x))
        x = self.fc_out(x)

        return x

In [100]:
import torchvision.transforms as T
from PIL import Image

class FusionDataset(torch.utils.data.Dataset):
    def __init__(self, visual_root, audio_root, split, num_frames=16, max_audio=16000*10):
        self.samples = []
        self.num_frames = num_frames
        self.max_audio = max_audio

        self.transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        classes = ["Adult Content", "Harmful Content", "Safe", "Suicide"]
        self.class_to_idx = {c: i for i, c in enumerate(classes)}

        for cls in classes:
            v_cls = os.path.join(visual_root, split, cls)
            a_cls = os.path.join(audio_root, split, cls)

            for vid in sorted(os.listdir(v_cls)):
                vid_path = os.path.join(v_cls, vid)

                if not os.path.isdir(vid_path):
                    continue

                audio_path = os.path.join(a_cls, vid + ".wav")

                if os.path.exists(audio_path):
                    self.samples.append((vid_path, audio_path, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        vid_path, audio_path, label = self.samples[idx]

        # ---- VIDEO ----
        frame_files = sorted(os.listdir(vid_path))

        if len(frame_files) >= self.num_frames:
            indices = torch.linspace(0, len(frame_files)-1, self.num_frames).long()
            selected = [frame_files[i] for i in indices]
        else:
            selected = frame_files + [frame_files[-1]] * (self.num_frames - len(frame_files))

        frames = []
        for f in selected:
            img = Image.open(os.path.join(vid_path, f)).convert("RGB")
            frames.append(self.transform(img))

        video = torch.stack(frames).permute(1, 0, 2, 3)

        # ---- AUDIO ----
        waveform, sr = torchaudio.load(audio_path)

        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)

        waveform = waveform.mean(dim=0)

        if waveform.shape[0] < self.max_audio:
            pad = self.max_audio - waveform.shape[0]
            waveform = F.pad(waveform, (0, pad))
        else:
            waveform = waveform[:self.max_audio]

        waveform = waveform.unsqueeze(0)  # (1, T)

        return video, waveform, label

In [101]:
val_dataset = FusionDataset(
    visual_root="/kaggle/input/datasets/aryansraut/preprocessed-ucf-crime-dataset-visual/TikHarm_frames_16",
    audio_root="/kaggle/input/datasets/arsalan9702/tickharm-pre-processed-audio/TikHarm_audio",
    split="val"
)

test_dataset = FusionDataset(
    visual_root="/kaggle/input/datasets/aryansraut/preprocessed-ucf-crime-dataset-visual/TikHarm_frames_16",
    audio_root="/kaggle/input/datasets/arsalan9702/tickharm-pre-processed-audio/TikHarm_audio",
    split="test"
)

val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

In [102]:
train_dataset = FusionDataset(
    visual_root="/kaggle/input/datasets/aryansraut/preprocessed-ucf-crime-dataset-visual/TikHarm_frames_16",
    audio_root="/kaggle/input/datasets/arsalan9702/tickharm-pre-processed-audio/TikHarm_audio",
    split="train"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [103]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Visual
visual_model = torch.hub.load(
    "pytorch/vision:v0.15.2",
    "swin3d_t",
    pretrained=False
)
visual_model.head = nn.Linear(visual_model.head.in_features, 4)

ckpt = torch.load(
    "/kaggle/input/models/arsalan9702/swin-3d/pytorch/default/1/best_swin3d_tikharm.pt",
    map_location=device
)

visual_model.load_state_dict(ckpt["model_state_dict"])
visual_model = visual_model.to(device)


# Audio
audio_model = CNN14(classes_num=4)

audio_model.load_state_dict(
    torch.load("/kaggle/input/models/arsalan9702/swin-3d-cnn-14/pytorch/default/1/best_audio_cnn14.pth")
)

audio_model = audio_model.to(device)

if torch.cuda.device_count() > 1:
    visual_model = nn.DataParallel(visual_model)
    audio_model = nn.DataParallel(audio_model)

visual_model.eval()
audio_model.eval()

Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.15.2


DataParallel(
  (module): CNN14(
    (spectrogram_extractor): Spectrogram(
      (stft): STFT(
        (conv_real): Conv1d(1, 513, kernel_size=(1024,), stride=(320,), bias=False)
        (conv_imag): Conv1d(1, 513, kernel_size=(1024,), stride=(320,), bias=False)
      )
    )
    (logmel_extractor): LogmelFilterBank()
    (bn0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv_block1): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (conv_block2): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, st

In [106]:
def evaluate_fusion(loader, alpha=0.8):
    correct = 0
    total = 0

    visual_model.eval()
    audio_model.eval()

    with torch.no_grad():
        for video, audio, labels in loader:
            video = video.to(device)
            audio = audio.to(device).squeeze(1)  
            labels = labels.to(device)

            v_logits = visual_model(video)
            a_logits = audio_model(audio)

            fused = alpha * v_logits + (1 - alpha) * a_logits
            preds = torch.argmax(fused, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [105]:
for alpha in [0.7, 0.8, 0.9]:
    val_acc = evaluate_fusion(val_loader, alpha)
    test_acc = evaluate_fusion(test_loader, alpha)

    print(f"Alpha: {alpha}")
    print(f"Val Acc:  {val_acc:.4f}")
    print(f"Test Acc: {test_acc:.4f}")
    print("-" * 30)

Alpha: 0.7
Val Acc:  0.8838
Test Acc: 0.8671
------------------------------
Alpha: 0.8
Val Acc:  0.8813
Test Acc: 0.8658
------------------------------
Alpha: 0.9
Val Acc:  0.8763
Test Acc: 0.8671
------------------------------


In [113]:
class FusionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 32),
            nn.ReLU(),
            nn.Linear(32, 4)
        )

    def forward(self, v_logits, a_logits):
        x = torch.cat([v_logits, a_logits], dim=1)
        return self.net(x)

In [118]:
fusion_model = FusionMLP().to(device)

for p in visual_model.parameters():
    p.requires_grad = False

for p in audio_model.parameters():
    p.requires_grad = False

optimizer = torch.optim.Adam(fusion_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [119]:
fusion_model.apply(lambda m: m.reset_parameters() if hasattr(m, 'reset_parameters') else None)

FusionMLP(
  (net): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=4, bias=True)
  )
)

In [120]:
from tqdm import tqdm

def train_fusion(loader):
    fusion_model.train()
    total_loss = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for video, audio, labels in pbar:
        video = video.to(device)
        audio = audio.to(device).squeeze(1)
        labels = labels.to(device)

        with torch.no_grad():
            v_logits = visual_model(video)
            a_logits = audio_model(audio)

        out = fusion_model(v_logits, a_logits)
        loss = criterion(out, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        pbar.set_postfix(loss=loss.item())

    return total_loss / len(loader)

In [121]:
def evaluate_fusion_mlp(loader):
    fusion_model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for video, audio, labels in loader:
            video = video.to(device)
            audio = audio.to(device).squeeze(1)
            labels = labels.to(device)

            v_logits = visual_model(video)
            a_logits = audio_model(audio)

            out = fusion_model(v_logits, a_logits)
            preds = torch.argmax(out, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [122]:
best_val_acc = 0
patience = 3
counter = 0

for epoch in range(10):
    loss = train_fusion(train_loader)

    val_acc = evaluate_fusion_mlp(val_loader)
    test_acc = evaluate_fusion_mlp(test_loader)

    print(f"Epoch {epoch+1}")
    print(f"Loss: {loss:.4f}")
    print(f"Val Acc:  {val_acc:.4f}")
    print(f"Test Acc: {test_acc:.4f}")
    print("-" * 30)

    # ---- EARLY STOPPING ----
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0

        # save best model
        torch.save(fusion_model.state_dict(), "best_fusion_mlp.pth")

    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered")
        break

Epoch 1
Loss: 0.3272
Val Acc:  0.8889
Test Acc: 0.8646
------------------------------


Epoch 2
Loss: 0.0331
Val Acc:  0.8864
Test Acc: 0.8658
------------------------------


Epoch 3
Loss: 0.0232
Val Acc:  0.8838
Test Acc: 0.8696
------------------------------


Epoch 4
Loss: 0.0200
Val Acc:  0.8838
Test Acc: 0.8684
------------------------------
Early stopping triggered


In [123]:
def confidence_metrics(logits):
    probs = F.softmax(logits, dim=1)

    confidence = probs.max(dim=1).values
    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1)

    return confidence, entropy

In [124]:
def evaluate_with_confidence(loader, alpha=0.8):
    all_conf, all_ent = [], []
    correct, total = 0, 0

    visual_model.eval()
    audio_model.eval()

    with torch.no_grad():
        for video, audio, labels in loader:
            video = video.to(device)
            audio = audio.to(device).squeeze(1)   
            labels = labels.to(device)

            v_logits = visual_model(video)
            a_logits = audio_model(audio)

            logits = alpha * v_logits + (1 - alpha) * a_logits

            probs = F.softmax(logits, dim=1)

            conf = probs.max(dim=1).values
            ent = -(probs * torch.log(probs + 1e-8)).sum(dim=1)

            preds = logits.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_conf.append(conf.cpu())
            all_ent.append(ent.cpu())

    all_conf = torch.cat(all_conf)
    all_ent = torch.cat(all_ent)

    print("Accuracy:", correct / total)
    print("Mean Confidence:", all_conf.mean().item())
    print("Mean Entropy:", all_ent.mean().item())

In [125]:
fusion_model.load_state_dict(torch.load("best_fusion_mlp.pth"))
fusion_model.eval()

final_test_acc = evaluate_fusion_mlp(test_loader)
print("Final Test Accuracy:", final_test_acc)

Final Test Accuracy: 0.8645569620253165


In [126]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np

def full_report(loader, model_type="fusion_mlp"):
    y_true = []
    y_pred = []

    fusion_model.eval()
    visual_model.eval()
    audio_model.eval()

    with torch.no_grad():
        for video, audio, labels in loader:
            video = video.to(device)
            audio = audio.to(device)
            labels = labels.to(device)

            # ---- FEATURES ----
            if model_type in ["feature", "transformer"]:
                vf = vfeat(video)
                af = afeat(audio)
                logits = fusion_model(vf, af)

            elif model_type == "mlp":
                audio = audio.squeeze(1)
                v_logits = visual_model(video)
                a_logits = audio_model(audio)
                logits = fusion_model(v_logits, a_logits)

            elif model_type == "alpha":
                audio = audio.squeeze(1)
                v_logits = visual_model(video)
                a_logits = audio_model(audio)
                logits = 0.8 * v_logits + 0.2 * a_logits

            preds = torch.argmax(logits, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    classes = ["Adult Content", "Harmful Content", "Safe", "Suicide"]

    print("\n=== Accuracy ===")
    print(accuracy_score(y_true, y_pred))

    print("\n=== Classification Report ===")
    print(classification_report(y_true, y_pred, target_names=classes))

    print("\n=== Confusion Matrix ===")
    print(confusion_matrix(y_true, y_pred))

In [127]:
# full_report(test_loader, model_type="feature")      # feature fusion
# full_report(test_loader, model_type="transformer") # transformer
full_report(test_loader, model_type="mlp")         # old MLP
# full_report(test_loader, model_type="alpha")       # alpha baseline


=== Accuracy ===
0.8645569620253165

=== Classification Report ===
                 precision    recall  f1-score   support

  Adult Content       0.86      0.90      0.88       195
Harmful Content       0.88      0.76      0.82       198
           Safe       0.89      0.88      0.88       200
        Suicide       0.83      0.93      0.88       197

       accuracy                           0.86       790
      macro avg       0.87      0.86      0.86       790
   weighted avg       0.87      0.86      0.86       790


=== Confusion Matrix ===
[[175   8   3   9]
 [ 17 150  14  17]
 [  5   9 175  11]
 [  6   3   5 183]]
